In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from model.actor_critic import FrameObservationEncoderNet, EncoderNet
from dataset import get_dataloader

from tqdm import trange
import tqdm

In [2]:
def cosine_loss(x, y):
    return 1. - F.cosine_similarity(x, y, dim=-1).mean()

def mse_loss(x, y):
    return F.mse_loss(x, y, reduction="mean")

In [3]:
class Alignment(nn.Module):
    def __init__(self, state_encoder, frame_encoder, state_feature_layer=-1):
        super().__init__()

        self.state_feature_layer = state_feature_layer

        self.state_encoder = state_encoder

        self.frame_encoder = frame_encoder

        self.state_encoder.eval()

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)

        return state_features[self.state_feature_layer]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames, False, False)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
encoder_weight, actor_weight, critic_wegit = torch.load("state_model.pth", weights_only=True)

state_encoder = EncoderNet(6+6+3+4+3+4, [256, 256, 256]).to(device)
frame_encoder = FrameObservationEncoderNet(6, state_encoder.dim).to(device)

state_encoder.load_state_dict(encoder_weight)

model = Alignment(state_encoder, frame_encoder, -1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=epochs)
dataloader = get_dataloader()
size = len(dataloader.dataset)

In [5]:
for _ in trange(epochs, desc="Epochs"):
    running_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        
        loss = 0.5 * mse_loss(frame_features, vector_features) + 0.5 * cosine_loss(frame_features, vector_features)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
    
    scheduler.step()
    train_loss = running_loss / size
    tqdm.tqdm.write(f"Train Loss: {train_loss:.4f}")

torch.save([model.frame_encoder.state_dict(), actor_weight, critic_wegit], "frame_model.pth")

Epochs:   5%|▌         | 1/20 [00:21<06:50, 21.59s/it]

Train Loss: 0.8126


Epochs:  10%|█         | 2/20 [00:42<06:20, 21.12s/it]

Train Loss: 0.5268


Epochs:  15%|█▌        | 3/20 [01:02<05:52, 20.72s/it]

...

Train Loss: 0.1171


Epochs:  95%|█████████▌| 19/20 [06:24<00:20, 20.08s/it]

Train Loss: 0.1153


Epochs: 100%|██████████| 20/20 [06:44<00:00, 20.23s/it]

Train Loss: 0.1144


In [6]:
cosine_loss(frame_features, vector_features)

tensor(0.0748, device='cuda:0', grad_fn=<RsubBackward1>)